# 320 · e-f Rabi as a workflow — solution

**Level:** advanced · **Topics:** Gen2, SHFQC, workflow, quantum-operations, applications-library, sweeps · **Type:** write the code

Solution of [exercise.ipynb](exercise.ipynb).

In [ ]:
import numpy as np

from laboneq import workflow
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation
from laboneq.dsl.quantum import QPU
from laboneq.simple import *
from laboneq.workflow.tasks import compile_experiment, run_experiment
from laboneq_applications.experiments.options import (
    TuneupExperimentOptions,
    TuneUpWorkflowOptions,
)
from laboneq_applications.qpu_types.tunable_transmon import (
    TunableTransmonOperations,
    TunableTransmonQubit,
)

## Device setup

`drive` and `drive_ef` are connected to the same port.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/drive_ef", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Qubit and QPU

There is no hand-written calibration. The frequencies, amplitudes and lengths are **qubit parameters**,
and `q0.calibration()` builds the calibration from them. The QPU holds the qubit and its quantum operations.

📖 [Quantum operations](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/04_quantum_processing_unit/tutorials/00_quantum_operations.html) ·
[Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html)

In [ ]:
[q0] = TunableTransmonQubit.from_device_setup(setup)
q0.update(
    drive_lo_frequency=4.0e9,
    resonance_frequency_ge=4.10e9,
    resonance_frequency_ef=3.93e9,
    ge_drive_amplitude_pi=0.7,
    ge_drive_length=23e-9,
    ef_drive_length=16e-9,
    readout_lo_frequency=7.0e9,
    readout_resonator_frequency=7.08e9,
    readout_amplitude=0.5,
    readout_length=2e-6,
    readout_integration_length=2e-6,
)

qpu = QPU(quantum_elements=[q0], quantum_operations=TunableTransmonOperations())
setup.set_calibration(q0.calibration())

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Step 1: the task that creates the experiment

`@dsl.qubit_experiment` takes the signals from the qubit: no signal list and no signal map.
`@workflow.task` makes the function usable in a workflow.

Two quantum operations are missing: bring the qubit to `e`, then play the e-f π pulse with the swept amplitude.

📖 [Tasks](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/07_workflow/tutorials/00_tasks.html) ·
[Writing experiments](https://docs.zhinst.com/labone_q_user_manual/applications_library/tutorials/sources/writing_experiments.html) ·
[Quantum operations tutorial](https://docs.zhinst.com/labone_q_user_manual/applications_library/tutorials/sources/quantum_operations.html)

In [ ]:
@workflow.task
@dsl.qubit_experiment
def create_experiment(qpu, qubit, amplitudes, options: TuneupExperimentOptions | None = None):
    if options is None:
        options = TuneupExperimentOptions()
    qop = qpu.quantum_operations
    amplitude = SweepParameter(uid="ef_amplitude", values=amplitudes)

    with dsl.acquire_loop_rt(
        count=options.count,
        averaging_mode=options.averaging_mode,
        acquisition_type=options.acquisition_type,
    ):
        with dsl.sweep(parameter=amplitude):
            qop.prepare_state(qubit, state="e")
            qop.x180(qubit, amplitude=amplitude, transition="ef")
            qop.measure(qubit, dsl.handles.result_handle(qubit.uid))
            qop.passive_reset(qubit)

## Step 2: the workflow

Three tasks in a row: create the experiment, compile it, run it. Then return the results.
`compile_experiment` and `run_experiment` are ready-made tasks.

📖 [Workflows](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/07_workflow/tutorials/01_workflows.html) ·
[Workflow syntax](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/07_workflow/tutorials/02_workflow_syntax.html)

In [ ]:
@workflow.workflow(name="ef_rabi")
def ef_rabi(session, qpu, qubit, amplitudes, options: TuneUpWorkflowOptions | None = None):
    experiment = create_experiment(qpu, qubit, amplitudes)
    compiled = compile_experiment(session, experiment)
    results = run_experiment(session, compiled)
    workflow.return_(results)

## Step 3: run

Calling the workflow does not run anything yet: it only builds it. `.run()` executes it.
The options of all tasks are collected in one object, here used to set the number of shots.
The `count` option only exists once the workflow of step 2 calls `create_experiment`.

📖 [Workflow options](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/07_workflow/tutorials/03_options.html)

In [ ]:
options = ef_rabi.options()
options.count(10)

my_workflow = ef_rabi(session, qpu, q0, np.linspace(0.1, 1.0, 10), options=options)
print(my_workflow.graph.tree)

workflow_result = my_workflow.run()

In [ ]:
# check
print([task.name for task in workflow_result.tasks])
print(workflow_result.output[dsl.handles.result_handle("q0")].data.shape)

**Expected result:**

```
['create_experiment', 'compile_experiment', 'run_experiment']
(10,)
```

Every task keeps its output. Take the compiled experiment from the `compile_experiment` task and look at the first shot:
the g-e pulse, the e-f pulse right after it, then the readout.

In [ ]:
compiled = workflow_result.tasks["compile_experiment"].output
plot_simulation(compiled, start_time=0, length=150e-9, signal_names_to_show=["drive", "measure"])

## Answers

1. Inside the quantum operations. Each operation is a section on the system grid, so the change of hardware oscillator between the g-e and the e-f pulse always has room.
2. From the qubit parameters (`ge_drive_amplitude_pi`, `ge_drive_length`, `ef_drive_length`, ...). The operations read them from the qubit, so a recalibration changes the parameters and not the experiment.
3. `options.count(1000)`. The workflow passes each option to the task that uses it, here `create_experiment`.
4. `amplitude_rabi`, with the option `transition="ef"`.

**Takeaway:** an experiment of the Applications Library is a task that creates the experiment from qubits and quantum operations, inside a workflow that compiles and runs it.

## The same experiment, from the library

`amplitude_rabi` with the option `transition="ef"` is this experiment, with analysis and parameter update on top.

📖 [Amplitude Rabi workflow](https://docs.zhinst.com/labone_q_user_manual/applications_library/how-to-guides/sources/01_superconducting_qubits/01_workflows/04_amplitude_rabi.html)

In [ ]:
from laboneq_applications.experiments import amplitude_rabi

library_options = amplitude_rabi.experiment_workflow.options()
library_options.transition("ef")
library_options.count(10)
library_options.do_analysis(False)

library_workflow = amplitude_rabi.experiment_workflow(
    session, qpu, "q0", amplitudes=np.linspace(0.1, 1.0, 10), options=library_options
)
print(library_workflow.graph.tree)
library_result = library_workflow.run()